In [ ]:
# Load packages and dataset.
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from numpy.testing import assert_allclose

In [ ]:
# Load the iris dataset as a bunch object, which is like a dictionary, with keys and attributes.
iris = load_iris(as_frame=True)
print(iris.keys())

In [ ]:
# Look inside the data to determine how to format for pandas.
# The data I need is inside the "frame" key.
for key in iris.keys():
    print("The contents of the '{}' key are:".format(key))
    print(iris[key])

In [ ]:
# Load the "frame" data as a dataframe.
df = iris["frame"]
df

PCA - Exploratory data analysis.

* Generally, whilst we may have some a priori hunches about data before looking at it, we need to determine if the data is amenable to PCA.
* The main EDA tools we often use for PCA are:
    * Pairs plots.
    * Correlation heat maps.

Pairs plots are generally used to plot bivariate scatter amongst pairs of features, and to get a sense of univariate distributions of each feature, with the view to searching for "clusters".

In [ ]:
eda_df = df.drop("target", axis="columns")
_ = sns.pairplot(eda_df)

In [ ]:
g = sns.PairGrid(eda_df)
g.map_upper(sns.scatterplot)
g.map_lower(sns.kdeplot)
g.map_diag(sns.histplot)

In [ ]:
# Plot correlation heatmap.

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(eda_df.corr(), annot=True, cmap="Blues", ax=ax)
ax.set_title("Correlation Matrix - Iris Dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Create an additional column of target names using a dictionary mapping class numbers to class names, and add this column to the end.
target_name_dict = {0: "Setosa", 1: "Versicolor", 2: "Virginica"}
target_names = df["target"].replace(target_name_dict)
print(target_names)
df["target names"] = target_names

In [ ]:
# The new dataframe with added columns.
df

In [ ]:
# Now visualise a pairs-plot, which plots data columns in pairs as scatter plots.
# To do this, remove the "target" class label column, otherwise these will be included in scatter plots.
pairs_df = df.drop("target", axis="columns")
_ = sns.pairplot(pairs_df, hue="target names")

* The diagonal plots here are kernel density estimates, akin to continuous plots of the probability density of the random variable in question.
    * Some clustering across classes in sepal length, going from lowest setosa, to versicolor, virginica highest.
    * Much more clustering across classes in sepal width, and all quite similar.
    * Petal measurements much more distinctly clustered than sepal measurements.
    * Very pronounced clustering across petal length, going from lowest setosal, to versicolor, viriginica highest.
    * Most pronounced clustering across petal width, going from lowest setosa, to versicolor, virginica highest. 
    * Kernel density estimates suggest,
        * Setosa - Very short petal widths and lengths, sepal lengths, and highest sepal width.
        * Versicolor - Average measurements across the board, lowest sepal width
        * Virginica - High petal widths, lengths, sepal lengths, average sepal width.
    * PCA will be looking at directions of greatest variance - can you identify which ones are going to be selected?
        * Not before standardisation.
* The off-diagonal plots are bivariate scatter plots
* Continue analysis tomorrow.

PCA - pre-processing.

- Theoretical accounts of PCA, both at the population level and at the sample level mandate mean-centering as an essential step in pre-processing. - why?
- But the case for standardisation in pre-processing, i.e. dividing by the standard deviation, or its sample estimate, is a matter of data analysis.
- If we are working with features with different measurements e.g. sepal lengths and also mineral uptake efficiency, then standardisation is required to homogenise variations in measurement magnitudes for comparison.
- As we are dealing with uniform measurements in centimetres, we decide to mean center but not scale the data by dividing by the standard deviation.

Data Leakage Warning (DLW)

- Although we are carrying out dimensionality reduction, we will not be using transformed as an input into a supervised learning model.
- If we were to do so, then carrying out PCA on the entire dataset without first splitting it into a training-test set would mean that information from the test-set would "leak" into the training phase.
- In this case, we would carry out our PCA pre-processing on the training data only, then use the same transform on the test-set data.

To anchor this in the theory at population level, we have a $d$-dimensional random vector $Y \in \mathbb{R}^d$ with mean $\mu_Y$. For PCA, we instead work with the mean-centred random vector $X \in \mathbb{R}^d$, where $X = Y - \mu_Y$, so that the mean of $X$ is $\mu = 0$, and its variance-covariance matrix is $\boldsymbol{\Sigma} = \text{Cov}(X) = \mathbb{E}[XX^{\top}]$.

Our iris-data consists of an observation matrix $\boldsymbol{Y} \in \mathbb{R}^{n \times d}$ with $n = 150$ observations and $d = 4$ features. This consists of the observation vectors $y_i \in \mathbb{R}^d$ of the $Y$ stored as rows $y_i^{\top}$ of this matrix. With the sample mean vector $\overline{y} = (1 / n)\sum^n_{i=1} y_i$, then we have that our mean-centred observation matrix is,

$$\mathbf{X}\in \mathbb{R}^{n \times d} = \mathbf{Y} - \bf{1}\overline{y}^T$$

In [ ]:
# Implement mean-subtraction using pandas inline functions.
# Compute the mean of each column, dropping the "target names" column as non-numeric, and dropping the "target" column containing flower class numbers.
df_numeric = df.select_dtypes(include=["number"])
df_numeric = df_numeric.iloc[:, 0:4]
df_numeric.mean()

In [ ]:
# Now implement mean-subtraction.
df_numeric - df_numeric.mean()

In [ ]:
# Now use sklearn preprocessing using sklearn.preprocessing.StandardScaler.
# Remember we set `with_std=False` kwarg because we do not want scaling, only centering.
scaler = StandardScaler(with_mean=True, with_std=False)
scaler = scaler.set_output(transform="pandas")
X = scaler.fit_transform(df_numeric)
X

PCA - some population-level theory.

At population level, PCA chooses a subspace $U = \text{span}(u_1, \dots, u_k)$, where $k \leq d$, so as to maximise the total variance of the projection of the random vector $X$ onto $U$. In coordinates, working in the standard basis and defining the orthonormal matrix as 

$$\boldsymbol{U} \in \mathbb{R}^{d \times k} = \begin{bmatrix} | &  & | \\ u_1 & \dots & u_k \\ | & & | \end{bmatrix},$$

then the projection of the random vector $X$ onto $U$, specified in subspace $U$ coordinates is,

$$P_UX = \boldsymbol{U}^{\top}X$$

The covariance matrix of the random projection $P_UX$ is,

$$\operatorname{Cov}(P_UX) = \mathbb{E}[\boldsymbol{U}^{\top}XX^{\top} \boldsymbol{U}] = \boldsymbol{U}^{\top}\mathbb{E}[XX^{\top}]\boldsymbol{U} = \mathbf{U}^{\top}\boldsymbol{\Sigma}\mathbf{U}.$$

The total variance of $P_UX$ is defined as the trace of this covariance matrix, and so at population level, PCA solves,

$$\boldsymbol{U}^* = \underset{u_1, \dots, u_k}{\operatorname{argmax}} \operatorname{tr}[\boldsymbol{U}^{\top}\boldsymbol{\Sigma} \boldsymbol{U}] \quad \text{ s.t. }  \quad\boldsymbol{U}^{\top}\boldsymbol{U} = \boldsymbol{I}.$$

The solution is given by the orthonormal matrix $\mathbf{U}^* = \mathbf{V}$, where the columns of $\mathbf{V}$, consist of $k$ eigenvectors $v_1, \dots, v_k$ corresponding to the $k$ largest eigenvalues $\lambda_1, \dots, \lambda_k$ of the covariance matrix $\boldsymbol{\Sigma}$.  Hence the total variance is maximised as the sum of eigenvalues of the covariance matrix $\boldsymbol{\Sigma}$.

$$\operatorname{tr}[{\mathbf{V}}^{\top}{\boldsymbol{\Sigma}}\mathbf{V}] = \sum^k_{i=1} \operatorname{Var}[v_i^{\top}X] = \sum^k_{i=1} \hat{v}_i^{\top} \boldsymbol{\Sigma}v_i = \sum^k_{i=1} v_i^{\top} \lambda_i v_i = 
\sum^k_{i=1} \lambda_i v_i^{\top}v_i = \sum^{k}_{i=1} \lambda_i$$

Where the first equality is conditional on our optimising over a subspace $U$ spanned by orthonormal basis vectors. This ensure that the total variance of the random projection $P_UX$ decomposes into an additive sum of variances of the scalar projections of $X$ onto each direction $u_i$, so that $\operatorname{tr}[{\mathbf{V}}^{\top}{\boldsymbol{\Sigma}}\mathbf{V}] = \sum^k_{i=1} \operatorname{Var}[v_i^{\top}X]$

The optimised vectors $u_1^* = v_1, \dots, u_k^* = v_k$ are known as the principal directions, and we use this terminology instead of "principal components". Confusingly, "principal components" in the literature can also to the projected data into subspace coordinates...

PCA - some sample-level theory.

Statistically, $\boldsymbol{\Sigma}$ is a fixed, unknown quantity, and similarly with $\operatorname{Var}[v_i^{\top}X] = \lambda_i$ At sample-level, we use the plug-in principle, replacing $\boldsymbol{\Sigma}$ with its estimator $\mathbf{S}$, the sample covariance matrix $\boldsymbol{S} = (n-1)^{-1} \sum^n_{i=1}x_i x_i^{\top}$, realised on our dataset.

 Correspondingly, $\hat{\mathbf{V}}$ is given by the top $k$ eigenvectors of the sample covariance matrix $\mathbf{S}$, and the total variance is maximised as the sum of eigenvalues $\hat{\lambda_i}$ of the sample covariance matrix $\mathbf{S}$. Where $\hat{\mathbf{V}}$ and $\hat{\lambda_i}$ are random, being functions of the random matrix $\boldsymbol{S}$.

PCA - choosing the number of components.

* The number of principal directions $k$ is the dimension of the subspace $U$ we wish to project onto, and represents the number of dimensions $k$ of the data we wish to reduce to from $d$.
* It can be viewed as a hyperparameter that is:
    * Chosen by means of an exploratory scree-plot with a heuristic "cut-off" for the proportion of variance explained by each principal component.
    * Tuned via cross-validation, typically used when using PCA to dimensionally reduce data for use in a downstream ML task in a pipeline e.g. as input to a neural network.
    * Selected via automatic algorithmic methods.
* As this is a toy dataset, we will go down the first route.
***
* Data Leakage Warning: Don't ever call `.fit()` or `.fit_transform()` on test data!
    - Context of this DLW is if we are using PCA dimensionality as part of a broader ML pipeline.
    - If we are splitting the data into training data and testing data, then we use training data for:
        - PCA preprocessing like scaling/centering.
        - Extracting orthonormal bases.
        - Reduce the dimensionality of the data by projecting using orthonormal bases to train the supervised learning algorithm.
    - During testing phase, when we test our ML pipeline, then we can only do the following using test data:
        - PCA preprocess the test data e.g. centering and scaling using only estimates from training.
        - Reduce the dimensionality of the test data by projecting using orthonormal bases learned during training.

In [ ]:
# Now run PCA with the number of components equal to the number of dimensions.
# Will call `.fit()`, so that I can access the explained variance ratio of each principal component.
# We can plot this in a scree plot to determine how many principal components to actually use.
pca_full = PCA(n_components=4).set_output(transform="pandas")
pca_full.fit(X)

Before we look at scree plots, we need to be clear on two metrics, the explained variance ratio of each principal direction, and the cumulative explained variance ratio of the top $k$ principal directions. The following results hold in the low-dimensional regime where the number of observations is greater than the number of features $n > d$.

* At population level, the total variance of $X$ across all $d$ directions is $\operatorname{tr}[\boldsymbol{\Sigma}] = \sum^d_{i=1} \lambda_i$.
* The maximised total variance of the projection of $X$ onto the optimal subspace spanned by the top $k$ eigenvectors is,
$$\operatorname{tr}[\mathbf{V}^{\top} \mathbf{S} \mathbf{V}] = \sum^k_{j=1} \operatorname{Var}[v_j^{\top}X] = \sum^k_{j=1} \lambda_j$$
* Individually, the $i$-th principal direction eigenvector makes a contribution $\operatorname{Var}[v_i^TX] = \lambda_i$ to the maximised total variance of the projection of $X$.

The explained variance ratio of the $i$-th principal direction, and the cumulative explained variance ratio of the top $k$ principal directions are then defined as,

$$\text{EVR}_i = r_i =  \frac{\operatorname{Var}[v_i^{\top}X]}{\operatorname{tr}[\boldsymbol{\Sigma}]} = \frac{v_i^{\top}\boldsymbol{\Sigma}v_i}{\operatorname{tr}[\boldsymbol{\Sigma}]} = \frac{\lambda_i}{\sum^d_{i=1} \lambda_i}$$

The cumulative explained variance ratio of the top $k$ principal directions is,

$$\text{CEVR}_k = \sum^k_{j=1} r_j = \frac{\operatorname{tr}[\mathbf{V}^{\top} \boldsymbol{\Sigma} \mathbf{V}]}{\operatorname{tr}[\boldsymbol{\Sigma}]} = \frac{\sum^k_{j=1}\operatorname{Var}[v_j^{\top}X]}{\operatorname{tr}[\boldsymbol{\Sigma}]} = \frac{\sum^k_{j=1} v_j^{\top}\boldsymbol{\Sigma}v_j}{\operatorname{tr}[\boldsymbol{\Sigma}]} = \frac{\sum^k_{j=1} \lambda_j}{\sum^d_{i=1} \lambda_i}$$

At sample-level, we now plug-in our estimate $\mathbf{S}$ into the population expressions, and we have,

$$\text{EVR}_i = \frac{\hat{v}_i^{\top}\mathbf{S}\hat{v}_i}{\operatorname{tr}[\mathbf{S}]}  = \frac{\hat{\lambda}_i}{\sum^d_{i=1} \hat{\lambda}_i}, \quad \text{CEVR}_k = \frac{\operatorname{tr}[\hat{\mathbf{V}}^{\top} \mathbf{S} \hat{\mathbf{V}}]}{\operatorname{tr}[\mathbf{S}]} = \frac{\sum^k_{j=1} \hat{v}_j^{\top}\mathbf{S}\hat{v}_j}{\operatorname{tr}[\mathbf{S}]} = \frac{\sum^k_{j=1} \hat{\lambda}_j}{\sum^d_{i=1} \hat{\lambda}_i}$$

In [ ]:
# Tabulate the explained variance ratio of all principal directions, and the cumulative explained variance ratios.
scree_df = pd.DataFrame({"Principal component": [f"PC{i+1}  " for i in range(4)],
                         "Explained variance ratio": pca_full.explained_variance_ratio_,
                         "Cumulative explained variance ratio": np.cumsum(pca_full.explained_variance_ratio_)})
scree_df

The table shows that the first two principal directions explain 97% of the total variance of the data, with $\text{CEVR}_2 \times 100 = 97.7685\%$. The rank-2 optimal subspace $U^*$ recovers $97.7\%$ of the maximum possible value of $\operatorname{tr}{\mathbf{S}}$, and the remaining $\approx2.3\%$ variance residing in the rank-2 orthogonal complement of $U^*$ is discarded.

In plain English, expressing the data with 2 latent features allows us to describe 97.7% of the variation in the data when expressed using all 4 iris flower features.


In [ ]:
# Show this as a scree plot.
fig, ax = plt.subplots(figsize=(7,4))
sns.lineplot(data=scree_df, x="Principal component", y="Explained variance ratio",
             marker="o", label="Individual", ax=ax)
sns.lineplot(data=scree_df, x="Principal component", y="Cumulative explained variance ratio",
             marker="o", label="Cumulative", ax=ax)
ax.set_title("Scree Plot")
ax.set_ylabel("Proportion of Variance Explained")
ax.set_ylim(0,1)
plt.tight_layout()
plt.show()

- Our scree plot 
- Heuristically, common thresholds are to choose $k$ so that the cumulative explained variance ratio is around 80- 95%.
- And similarly, one looks for "elbows" on the scree-plot, showing that 

Running PCA - fitting the model and computing the loading matrix.

* Our optimal principal subspace $U^*$ will be spanned by the top $k$ eigenvectors of the sample covariance matrix $\mathbf{S}$.
* In matrix form, with $k = 2$, then our optimal orthornormal matrix $\mathbf{U}^* \in \mathbb{R}^{d \times k}$ will be the matrix $\hat{\mathbf{V}}$ with the top 2 eigenvectors as columns.
* In data-science speak, $\mathbf{V}$ is known as a "loading matrix", from now on we simply write it as $\mathbf{V}$, as it is now an estimate realised on our dataset.
* We now compute the loadings matrix as follows,

In [ ]:
# Fits PCA using a 2-dimensional principal subspace.
pca = PCA(n_components=2).set_output(transform="pandas")
pca.fit(X)

In [ ]:
# `.components_` shows each orthonormal basis vector as a row of matrix.
# We rewrite it so that the orthonormal basis matrix has each subspace basis vector as a column.
basis_output = pca.components_
V = basis_output.T
print(V)

As a quick numerical analysis sanity check, we can check that the loadings matrix $\mathbf{V}$ is indeed orthornormal, so that $\mathbf{V}^{\top}\mathbf{V} = \mathbf{I}$. To do so we use an absolute tolerance of `atol` = $10^{-10}$, and conduct the check,

$$\vert \mathbf{V}^{\top}\mathbf{V}_{ij} - \mathbf{I}_{ij} \vert \leq \texttt{atol}.$$

In [ ]:
# Check orthornormality of loadings matrix V.

VTV = V.T @ V
I_k = np.eye(pca.n_components_)

try:
    assert_allclose(VTV, I_k, atol=1e-10, rtol=0)
    print(f"Orthornormality confirmed V^T V = I")
except AssertionError as e:
    print(f"Orthonormality check failed: {e}")


At this stage there are two visualisations we can use to help us:

* Heatmaps of the loading matrix $V$.
* Biplots.

In [ ]:
# V_df = pd.DataFrame(V, columns=["1st principal direction, v1", "2nd principal direction, v2"], index=X.columns)
fig, ax = plt.subplots()
sns.heatmap(V, annot=True, cmap="Blues", yticklabels=X.columns, ax=ax)
ax.set_yticklabels(X.columns)
ax.set_xticklabels(["$v_1$", "$v_2$"])
ax.set_title("Loading matrix $\\mathbf{V}$")

PCA - projecting the data.

At population level, the projection of the random vector $X$ onto a $k$-dimensional subspace $U$ spanned by an orthonormal basis $u_1, \dots, u_k$ is given by,

$$P_UX = \langle X, u_1 \rangle u_1 + \dots + \langle X, u_k \rangle u_k.$$

We can read off the coordinates of $P_UX$ in the subspace basis. Using the standard basis $E$ as our column vector reference basis, and as we are projecting onto the principal subspace $\mathbf{U}^* = \mathbf{V}$, the projection of the random vector $X$ onto the principal subspace $\mathbf{V}$, in subspace coordinates, is the random column vector $p_V \in \mathbb{R}^{k}$,

$$p_{V} = \mathbf{V}^{\top}X = \begin{bmatrix} v_1^{\top} X \\ \vdots \\ v_k^{\top} X\end{bmatrix}.$$

At sample level, the projection of the $i$-th observation vector is then,

$$p_V^{(i)} = \mathbf{V}^{\top}x_i$$

Noting row conventions for the observation data matrix $\mathbf{X}$ and using the row-vector matrix multiplication view of matrix multiplication, the dimensionally reduced observation data matrix is then, $\mathbf{X} \mathbf{V} \in \mathbb{R}^{n \times k}$. Each row of this matrix consists of the projected observation vectors $(p^{(i)}_V)^{\top} = x_i^{\top} \mathbf{V}$, and this is computed below,

In [ ]:
# `.fit_transform() computes the dimensionally reduced projected data, in subspace coordinates.
proj_X = pca.fit_transform(X)
proj_X.columns = ["PC1", "PC2"]
proj_X

Again using our expression for the projection of the random vector $X$ onto the subspace $U$, 

$$P_UX = \langle X, u_1 \rangle u_1 + \dots + \langle X, u_k \rangle u_k.$$

We can also express this in $\mathbb{R}^d$ coordinates. Using the standard basis $E$ as our reference basis, replacing all inner products with Euclidean dot products gives, and using the linear combination view of matrix vector multiplication, we have

$$p_E = \mathbf{V}\mathbf{V}^{\top}X.$$

At sample-level, then

In [ ]:
# `.inverse_transform(proj_X)` computes a reconstruction of the projected data in the original space vector space coordinates.
reconstr_X = pca.inverse_transform(proj_X)
reconstr_X = pd.DataFrame(data=reconstr_X, columns=X.columns)
reconstr_X